### RAG Pipeline- Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\PROJECT\AppData\Local\Temp\ipykernel_19680\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [ ]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively (sorted, so the chunk order is deterministic)
    pdf_files = sorted(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            # Extend ONCE per file, outside the per-page loop above.
            # It used to sit inside that loop, so an n-page PDF added its pages n times.
            all_documents.extend(documents)
            print(f" \u2713 Loaded {len(documents)} pages")

        except Exception as e:
            print(f" \u2717 Error : {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-10-24T13:03:07+00:00', 'author': 'PROJECT', 'moddate': '2026-10-24T13:03:07+00:00', 'source': '..\\data\\pdf\\Java.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Java.pdf', 'file_type': 'pdf'}, page_content='What is Java? \nJava is a popular and powerful programming language, created in 1995.  \nIt is owned by Oracle, and more than 3 billion devices run Java. \nIt is used for: \n\uf0b7 Mobile applications (specially Android apps)  \n\uf0b7 Desktop applications \n\uf0b7 Web applications \n\uf0b7 Web servers and application servers  \n\uf0b7 Games \n\uf0b7 Database connection \n\uf0b7 And much, much more! \nWhy Use Java? \n\uf0b7 Java works on different platforms (Windows, Mac, Linux, Raspberry Pi, etc.)  \n\uf0b7 It is one of the most popular programming languages in the world  \n\uf0b7 It has a large demand in the current job market \n\uf0b7 It is easy to 

In [4]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [5]:
chunks=split_documents(all_pdf_documents)
chunks

Split 2 documents into 5 chunks

Example chunk:
Content: What is Java? 
Java is a popular and powerful programming language, created in 1995.  
It is owned by Oracle, and more than 3 billion devices run Java. 
It is used for: 
 Mobile applications (special...
Metadata: {'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-10-24T13:03:07+00:00', 'author': 'PROJECT', 'moddate': '2026-10-24T13:03:07+00:00', 'source': '..\\data\\pdf\\Java.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Java.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-10-24T13:03:07+00:00', 'author': 'PROJECT', 'moddate': '2026-10-24T13:03:07+00:00', 'source': '..\\data\\pdf\\Java.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Java.pdf', 'file_type': 'pdf'}, page_content='What is Java? \nJava is a popular and powerful programming language, created in 1995.  \nIt is owned by Oracle, and more than 3 billion devices run Java. \nIt is used for: \n\uf0b7 Mobile applications (specially Android apps)  \n\uf0b7 Desktop applications \n\uf0b7 Web applications \n\uf0b7 Web servers and application servers  \n\uf0b7 Games \n\uf0b7 Database connection \n\uf0b7 And much, much more! \nWhy Use Java? \n\uf0b7 Java works on different platforms (Windows, Mac, Linux, Raspberry Pi, etc.)  \n\uf0b7 It is one of the most popular programming languages in the world  \n\uf0b7 It has a large demand in the current job market \n\uf0b7 It is easy to 

### embedding And vectorStoreDB

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple


In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Vectors are unit length (normalize_embeddings=True). That is what lets us
        turn Chroma's L2 distance back into a cosine similarity (d = 2 - 2*cos).

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        if not texts:
            return np.empty((0, self.model.get_embedding_dimension()), dtype=np.float32)

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(
            list(texts), show_progress_bar=True, normalize_embeddings=True
        )
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager


### VectorStore

In [ ]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """

        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            # count() has to be CALLED: {self.collection.count} printed a bound method
            print(f"Existing documents in collection: {self.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def count(self) -> int:
        """Number of documents in the collection"""
        return self.collection.count()

    @property
    def space(self) -> str:
        """Distance space the collection actually uses ("l2" unless you set otherwise)"""
        configuration = getattr(self.collection, "configuration", None) or {}
        return (configuration.get("hnsw") or {}).get("space", "l2")

    def cosine_similarity(self, distance: float) -> float:
        """Turn a Chroma DISTANCE into a cosine similarity.

        Chroma returns distances, and what they mean depends on the collection's
        space. For unit-length embeddings:
          l2 (the default): distance = 2 - 2*cos  ->  cos = 1 - distance / 2
          cosine:           distance = 1 - cos    ->  cos = 1 - distance

        This notebook used to compute `1 - distance` on the default l2 space, which
        is 2*cos - 1: a chunk at cos 0.78 scored 0.57, off-topic chunks scored
        negative, and so any threshold silently discarded real context.
        """
        if self.space == "l2":
            return 1.0 - distance / 2.0
        if self.space == "cosine":
            return 1.0 - distance
        if self.space == "ip":
            return -distance
        raise ValueError(f"unsupported distance space: {self.space!r}")

    @staticmethod
    def _document_id(document, index: int) -> str:
        """Deterministic id: re-ingesting the same PDF updates in place.

        Deliberately NOT derived from the content. If it were, editing a PDF would
        mint new ids and leave the old chunks behind, and the collection would grow
        on every run again. Position-keyed ids overwrite in place; call reset() when
        a document shrinks, to drop the orphaned tail.
        """
        source = document.metadata.get('source_file') or document.metadata.get('source', '')
        page = document.metadata.get('page', '?')
        return f"doc_{source}_p{page}_c{index}"

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Uses upsert with deterministic ids, so running this cell again does not
        duplicate anything. It used to append on every run, which is how one
        collection ended up holding 5295 rows of near-identical chunks.

        Args:
            documents: List of Langchain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match exact number of embeddings")
        if not documents:
            print("Nothing to add: no documents supplied")
            return

        print(f"Adding {len(documents)} documents to vector store...")

        #Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Deterministic id, stable across runs
            ids.append(self._document_id(doc, i))

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.upsert(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

    def reset(self):
        """Delete and recreate the collection.

        Run this when the collection was built by an older version of this notebook:
        those stores hold thousands of duplicate chunks that crowd out good hits.
        """
        self.client.delete_collection(self.collection_name)
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "PDF document embeddings for RAG"}
        )
        print(f"Vector store reset. Documents in collection: {self.count()}")


vectorstore=VectorStore()
vectorstore


In [22]:
chunks

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-10-24T13:03:07+00:00', 'author': 'PROJECT', 'moddate': '2026-10-24T13:03:07+00:00', 'source': '..\\data\\pdf\\Java.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Java.pdf', 'file_type': 'pdf'}, page_content='What is Java? \nJava is a popular and powerful programming language, created in 1995.  \nIt is owned by Oracle, and more than 3 billion devices run Java. \nIt is used for: \n\uf0b7 Mobile applications (specially Android apps)  \n\uf0b7 Desktop applications \n\uf0b7 Web applications \n\uf0b7 Web servers and application servers  \n\uf0b7 Games \n\uf0b7 Database connection \n\uf0b7 And much, much more! \nWhy Use Java? \n\uf0b7 Java works on different platforms (Windows, Mac, Linux, Raspberry Pi, etc.)  \n\uf0b7 It is one of the most popular programming languages in the world  \n\uf0b7 It has a large demand in the current job market \n\uf0b7 It is easy to 

In [ ]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

## Rebuild the collection from ../data/pdf first, so re-running this notebook
## never accumulates stale or duplicated chunks.
vectorstore.reset()

##store in the vector database
vectorstore.add_documents(chunks,embeddings)


### Retriever Pipeline from VectorStore

In [ ]:
from typing import List, Dict, Any

class RAGRetriever:
    """Handles query-based retrieval from the vector store."""

    def __init__(self, vector_store, embedding_manager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:
        """Return the top-k chunks whose cosine similarity clears score_threshold."""

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        try:
            # 1. Generate query embedding
            query_embedding = self.embedding_manager.generate_embeddings(
                [query]
            )[0]

            # 2. Search in the vector store
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # 3. Process results
            retrieved_docs = []

            documents = results.get("documents") or [[]]
            metadatas = results.get("metadatas") or [[]]
            distances = results.get("distances") or [[]]
            ids = results.get("ids") or [[]]

            if documents[0]:
                for rank, (doc_id, document, metadata, distance) in enumerate(
                    zip(
                        ids[0],
                        documents[0],
                        metadatas[0],
                        distances[0]
                    ),
                    start=1
                ):
                    # Convert the distance to a real cosine similarity, otherwise
                    # the threshold compares apples to oranges (see VectorStore).
                    similarity_score = self.vector_store.cosine_similarity(distance)

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": rank
                        })

            print(
                f"Retrieved {len(retrieved_docs)} documents "
                "(after filtering)"
            )
            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


# Initialize the retriever
rag_retriever = RAGRetriever(vectorstore, embedding_manager)


In [25]:
rag_retriever

In [ ]:
rag_retriever.retrieve("Why Python")

In [ ]:
rag_retriever.retrieve("Java")



### Integration Vectordb Context pipeline With LLM output

In [ ]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from pathlib import Path
from dotenv import load_dotenv

## Read GROQ_API_KEY from the repo-root .env (this notebook sits in notebook/).
## load_dotenv() with no argument also picks up a notebook/.env if you prefer one.
load_dotenv(Path("../.env"))
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise RuntimeError(
        "GROQ_API_KEY is not set. Add a line GROQ_API_KEY=your-key-here to the "
        "repo's .env file (the one next to the notebook/ folder)."
    )

### Initialize the Groq LLM
## gemma2-9b-it was decommissioned by Groq (it now answers with a 404
## model_not_found), so this uses a model that is currently live.
llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-20b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retrieve the context
    results=retriever.retrieve(query,top_k=top_k)
    ## the retriever returns {'content': ...}; reading doc['context'] raised
    ## KeyError, which is why no context ever reached the LLM.
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."

    ## generate the answer using GROQ LLM
    prompt=f"""Use the following context to answer thwe question concisely.
        Context:
        {context}

        Question: {query}

        Answer: """

    response=llm.invoke(prompt)
    ## A LangChain AIMessage carries the answer on .content; there is no .context
    return response.content


In [ ]:
answer=rag_simple("What is Python?",rag_retriever,llm)
print(answer)

### Enhanced RAG Pipeline Features

In [ ]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}

    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer. The f-string already substitutes everything, so the stray
    # .format() this used to call crashed whenever the context contained braces.
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke(prompt)

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is Python?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])
